# EduPredict: Educational Analytics & Big Data Processing Notebook
**Aptech eProject Big Data & Machine Learning Pipeline**

This notebook provides an **end-to-end demonstration** of the EduPredict platform:
1. **Data Ingestion** - Generate 5,000 synthetic student records (Demographics, Academic, LMS, Attendance)
2. **Data Preprocessing** - Merge, clean (missing values, feature engineering, risk labeling)
3. **PySpark Distributed Processing** - HDFS partitioning & course demand aggregation
4. **Machine Learning Training** - GPA Predictor, Dropout Risk Classifier, Anomaly Detection
5. **Data Visualization** - Interactive plots & insights

> **Google Colab Ready** - Run cells in order for a complete pipeline.

## Step 0: Install Dependencies
Installs PySpark for distributed processing along with the ML/data science stack.

In [ ]:
!pip install -q pyspark pandas numpy scikit-learn joblib matplotlib seaborn
print("Dependencies installed.")

## Step 1: Imports & Environment Setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, IsolationForest
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report

print("All libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Step 2: Data Ingestion
Generates realistic synthetic datasets across 4 academic data sources for 5,000 students.

In [ ]:
num_students = 5000
seed = 42
np.random.seed(seed)
random.seed(seed)

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

student_ids = [f"STU{10000 + i}" for i in range(num_students)]

# --- 1. Demographics ---
demographics_df = pd.DataFrame({
    "student_id": student_ids,
    "age": np.random.randint(18, 26, size=num_students),
    "gender": np.random.choice(["Male", "Female", "Other"], size=num_students, p=[0.49, 0.49, 0.02]),
    "socio_economic_tier": np.random.choice(["Low", "Medium", "High"], size=num_students, p=[0.30, 0.50, 0.20]),
    "distance_from_campus_km": np.round(np.random.exponential(scale=9.5, size=num_students), 1),
    "internet_access": np.random.choice(["High-Speed", "Moderate", "Limited"], size=num_students, p=[0.60, 0.30, 0.10]),
    "parent_education": np.random.choice(["High School", "Bachelors", "Masters", "Doctorate"], size=num_students, p=[0.40, 0.40, 0.15, 0.05])
})

# --- 2. LMS Engagement ---
lms_df = pd.DataFrame({
    "student_id": student_ids,
    "lms_logins_per_week": np.random.poisson(lam=12, size=num_students),
    "video_watch_hours": np.round(np.random.normal(loc=15, scale=5, size=num_students).clip(1, 40), 1),
    "forum_posts": np.random.poisson(lam=4, size=num_students),
    "quiz_attempts": np.random.randint(2, 15, size=num_students),
    "avg_quiz_score": np.round(np.random.normal(loc=72, scale=12, size=num_students).clip(30, 100), 1)
})

# --- 3. Attendance ---
total_classes = np.random.choice([40, 45, 50], size=num_students)
attendance_rate = (0.5 * (lms_df["lms_logins_per_week"] / 20.0) +
                  0.5 * np.random.beta(a=5, b=2, size=num_students)).clip(0.3, 1.0)
attendance_rate = np.round(attendance_rate * 100, 1)
classes_attended = np.round((attendance_rate / 100.0) * total_classes).astype(int)

attendance_df = pd.DataFrame({
    "student_id": student_ids,
    "total_classes": total_classes,
    "classes_attended": classes_attended,
    "attendance_rate": attendance_rate,
    "tardy_count": np.random.poisson(lam=3, size=num_students)
})

# --- 4. Academic Records ---
base_score = (0.4 * attendance_rate) + (0.4 * lms_df["avg_quiz_score"]) + np.random.normal(loc=0, scale=8, size=num_students)
midterm_score = np.round(base_score.clip(25, 100), 1)
assignment_score = np.round((base_score + np.random.normal(0, 5, num_students)).clip(30, 100), 1)
final_score = np.round((0.3 * midterm_score + 0.3 * assignment_score + 0.4 * base_score).clip(20, 100), 1)
gpa = np.round((final_score / 100.0) * 4.0, 2)

academic_df = pd.DataFrame({
    "student_id": student_ids,
    "course_id": np.random.choice(["CS101", "DS201", "AI301", "DB401", "SE501"], size=num_students),
    "term": np.random.choice(["Fall 2025", "Spring 2026"], size=num_students),
    "midterm_score": midterm_score,
    "assignment_score": assignment_score,
    "final_score": final_score,
    "gpa": gpa,
    "credits": np.random.choice([3, 4], size=num_students)
})

# Save raw datasets
demographics_df.to_csv("data/raw/student_demographics.csv", index=False)
lms_df.to_csv("data/raw/lms_engagement.csv", index=False)
attendance_df.to_csv("data/raw/attendance_records.csv", index=False)
academic_df.to_csv("data/raw/academic_records.csv", index=False)

print(f"[OK] Generated {num_students} student records across 4 data sources")
print("Files written to data/raw/")

### Inspect the Raw Datasets

In [ ]:
print("=== Demographics ===")
display(demographics_df.head())
print(f"\n=== LMS Engagement ===")
display(lms_df.head())
print(f"\n=== Attendance ===")
display(attendance_df.head())
print(f"\n=== Academic Records ===")
display(academic_df.head())

print(f"\nTotal students: {len(student_ids)}")
print(f"Courses: {academic_df['course_id'].unique()}")

## Step 3: Data Preprocessing & Feature Engineering
Merges the 4 datasets, handles missing values, and computes derived features.

In [ ]:
df = academic_df.merge(demographics_df, on="student_id", how="inner")
df = df.merge(lms_df, on="student_id", how="inner")
df = df.merge(attendance_df, on="student_id", how="inner")
print(f"Merged dataset shape: {df.shape}")

# Handle missing values (median/mean imputation)
df.fillna({
    "midterm_score": df["midterm_score"].median(),
    "final_score": df["final_score"].median(),
    "attendance_rate": df["attendance_rate"].mean(),
    "lms_logins_per_week": 0,
    "avg_quiz_score": df["avg_quiz_score"].mean()
}, inplace=True)

# Feature engineering: Composite engagement index
df["engagement_index"] = np.round(
    (0.5 * df["attendance_rate"]) +
    (0.3 * (df["lms_logins_per_week"] / 20.0 * 100).clip(0, 100)) +
    (0.2 * df["avg_quiz_score"]),
    2
)

# Label dropout risk based on academic thresholds
conditions = [
    (df["attendance_rate"] < 60) | (df["gpa"] < 1.8) | (df["engagement_index"] < 45),
    (df["attendance_rate"] < 75) | (df["gpa"] < 2.5) | (df["engagement_index"] < 65)
]
choices = ["High", "Medium"]
df["dropout_risk"] = np.select(conditions, choices, default="Low")

# Anomaly labeling
df["is_anomaly"] = (
    ((df["lms_logins_per_week"] >= 10) & (df["final_score"] < 45)) |
    ((df["attendance_rate"] >= 85) & (df["avg_quiz_score"] < 40)) |
    (df["tardy_count"] >= 8)
)

output_path = "data/processed/edupredict_master_clean.csv"
df.to_csv(output_path, index=False)

print(f"[OK] Cleaned master dataset saved to {output_path}")
print(f"Total records: {len(df)}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Dropout risk distribution:\n{df['dropout_risk'].value_counts()}")
print(f"Anomalies: {df['is_anomaly'].sum()}")

### Exploratory Data Analysis & Visualization

In [ ]:
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 5)

# Dropout risk distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x="dropout_risk", hue="dropout_risk", palette="viridis", ax=axes[0], legend=False)
axes[0].set_title("Dropout Risk Distribution")
axes[0].set_xlabel("Risk Level")

df.groupby("course_id")["gpa"].mean().plot(kind="bar",
    color=["#6366f1", "#10b981", "#f59e0b", "#ef4444", "#8b5cf6"], ax=axes[1])
axes[1].set_title("Average GPA by Course")
axes[1].set_ylabel("Mean GPA")
axes[1].set_ylim(2.5, 3.5)

sns.histplot(df, x="gpa", hue="dropout_risk", multiple="stack", palette="viridis", ax=axes[2])
axes[2].set_title("GPA Distribution by Risk Level")

plt.tight_layout()
plt.show()



In [ ]:
# Correlation heatmap of numeric features
numeric_cols = ["age", "distance_from_campus_km", "midterm_score", "assignment_score",
                "final_score", "gpa", "lms_logins_per_week", "video_watch_hours",
                "forum_posts", "quiz_attempts", "avg_quiz_score", "attendance_rate",
                "tardy_count", "engagement_index"]

plt.figure(figsize=(14, 10))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# Key insight: attendance correlates strongly with GPA
print(f"Attendance vs GPA correlation: {df['attendance_rate'].corr(df['gpa']):.3f}")
print(f"LMS Logins vs GPA correlation: {df['lms_logins_per_week'].corr(df['gpa']):.3f}")

## Step 4: PySpark Distributed Processing
Simulates HDFS distributed processing. In Colab, Apache Spark is configured for local mode; on a real Hadoop cluster this code scales to petabytes.

In [ ]:
# Initialize Spark Session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("EduPredict_HDFS_Analytics") \
    .master("local[*]") \
    .getOrCreate()

spark_df = spark.read.csv("data/processed/edupredict_master_clean.csv", header=True, inferSchema=True)
print(f"Spark DataFrame partitions: {spark_df.rdd.getNumPartitions()}")
print(f"Total records: {spark_df.count()}")
spark_df.printSchema()

In [ ]:
# 1. Course Demand Aggregation (PySpark)
course_demand = spark_df.groupBy("course_id").agg(
    F.count("student_id").alias("total_enrolled"),
    F.avg("gpa").alias("avg_gpa"),
    F.avg("attendance_rate").alias("avg_attendance"),
    F.avg("engagement_index").alias("avg_engagement")
).orderBy(F.desc("total_enrolled"))

print("=== Course Demand Summary ===")
course_demand.show()

course_demand_pd = course_demand.toPandas()

In [ ]:
# 2. Feature Correlations (distributed)
correlations = {
    "attendance_vs_gpa": spark_df.stat.corr("attendance_rate", "gpa"),
    "lms_logins_vs_gpa": spark_df.stat.corr("lms_logins_per_week", "gpa"),
    "quiz_score_vs_final": spark_df.stat.corr("avg_quiz_score", "final_score")
}

print("=== Feature Correlations ===")
for k, v in correlations.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# 3. HDFS Partitioning: partition by dropout_risk for scalable storage
hdfs_dir = "data/processed/hdfs_partitioned_output"
spark_df.write.mode("overwrite").partitionBy("dropout_risk").parquet(f"{hdfs_dir}/risk_partitions")
print(f"[OK] HDFS partitioned output written to {hdfs_dir}/risk_partitions/")

# Verify partitioning
print("\n=== Partitioned Directories ===")
for d in sorted(os.listdir(f"{hdfs_dir}/risk_partitions")):
    print(d)

spark.stop()

In [ ]:
# Visualize the PySpark aggregation results
plt.figure(figsize=(12, 5))
sns.barplot(data=course_demand_pd, x="course_id", y="total_enrolled",
            hue="course_id", palette="viridis", legend=False)
plt.title("Course Enrollment Demand (PySpark Aggregation)")
plt.xlabel("Course")
plt.ylabel("Number of Enrolled Students")
for i, v in enumerate(course_demand_pd["total_enrolled"]):
    plt.text(i, v + 20, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()


## Step 5: Machine Learning Model Training
Trains three models: **GPA Predictor**, **Dropout Risk Classifier**, and **Anomaly Detector**.

In [ ]:
# Prepare features
feature_cols = [
    "age", "distance_from_campus_km", "midterm_score", "assignment_score",
    "lms_logins_per_week", "video_watch_hours", "forum_posts", "quiz_attempts",
    "avg_quiz_score", "total_classes", "classes_attended", "attendance_rate",
    "tardy_count", "engagement_index"
]

os.makedirs("models", exist_ok=True)

X = df[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features: {feature_cols}")


In [ ]:
# --- Model 1: Student Performance (GPA) Predictor ---
y_gpa = df["gpa"]
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_gpa, test_size=0.2, random_state=42)

perf_model = RandomForestRegressor(n_estimators=100, random_state=42)
perf_model.fit(X_train, y_train)

gpa_preds = perf_model.predict(X_test)
r2 = r2_score(y_test, gpa_preds)
rmse = np.sqrt(mean_squared_error(y_test, gpa_preds))

print(f"=== Student Performance (GPA) Predictor ===")
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

joblib.dump(perf_model, "models/performance_model.joblib")

In [ ]:
# --- Model 2: Dropout Risk Classifier ---
le_risk = LabelEncoder()
y_risk = le_risk.fit_transform(df["dropout_risk"])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_scaled, y_risk, test_size=0.2, random_state=42)
dropout_model = RandomForestClassifier(n_estimators=100, random_state=42)
dropout_model.fit(X_train_r, y_train_r)

risk_preds = dropout_model.predict(X_test_r)
acc = accuracy_score(y_test_r, risk_preds)

print(f"=== Dropout Risk Classifier ===")
print(f"Accuracy: {acc * 100:.2f}%")
print(classification_report(y_test_r, risk_preds,
    target_names=le_risk.classes_))

joblib.dump(dropout_model, "models/dropout_model.joblib")
joblib.dump(le_risk, "models/label_encoder_risk.joblib")

In [ ]:
# --- Model 3: Academic Anomaly Detector ---
anomaly_detector = IsolationForest(contamination=0.05, random_state=42)
anomaly_detector.fit(X_scaled)

anomaly_preds = anomaly_detector.predict(X_scaled)
n_anomalies = (anomaly_preds == -1).sum()

print(f"=== Academic Anomaly Detector (Isolation Forest) ===")
print(f"Detected anomalies: {n_anomalies} / {len(X_scaled)} ({n_anomalies/len(X_scaled)*100:.1f}%)")

joblib.dump(anomaly_detector, "models/anomaly_detector.joblib")
joblib.dump(scaler, "models/scaler.joblib")
joblib.dump(feature_cols, "models/feature_cols.joblib")
print("[OK] All models saved to models/")

## Step 6: Model Inference & Live Prediction Demo

In [ ]:
# Reload saved models for inference demo
scaler = joblib.load("models/scaler.joblib")
feature_cols = joblib.load("models/feature_cols.joblib")
perf_model = joblib.load("models/performance_model.joblib")
dropout_model = joblib.load("models/dropout_model.joblib")
le_risk = joblib.load("models/label_encoder_risk.joblib")
anomaly_detector = joblib.load("models/anomaly_detector.joblib")

def predict_student(sample):
    """Predict GPA, dropout risk, and anomaly status for a student."""
    row_values = [sample.get(col, 0.0) for col in feature_cols]
    X_input = pd.DataFrame([row_values], columns=feature_cols)
    X_scaled = scaler.transform(X_input)

    pred_gpa = max(0.0, min(4.0, float(perf_model.predict(X_scaled)[0])))
    risk_encoded = dropout_model.predict(X_scaled)[0]
    risk_label = le_risk.inverse_transform([risk_encoded])[0]
    risk_probs = dropout_model.predict_proba(X_scaled)[0]
    confidence = float(np.max(risk_probs))
    anomaly = bool(int(anomaly_detector.predict(X_scaled)[0]) == -1)

    return {
        "predicted_gpa": round(pred_gpa, 2),
        "dropout_risk": risk_label,
        "confidence_score": round(confidence, 2),
        "is_anomaly": anomaly
    }

# Test on a high-risk student profile (low attendance, low quiz, fewer LMS logins)
sample_student = {
    "age": 20, "distance_from_campus_km": 6.5, "midterm_score": 52.0,
    "assignment_score": 58.0, "lms_logins_per_week": 4, "video_watch_hours": 6.0,
    "forum_posts": 1, "quiz_attempts": 3, "avg_quiz_score": 60.0,
    "total_classes": 50, "classes_attended": 28, "attendance_rate": 56.0,
    "tardy_count": 7, "engagement_index": 45.0
}

result = predict_student(sample_student)
print("=== Prediction Result for High-Risk Student ===")
for k, v in result.items():
    print(f"{k}: {v}")

In [ ]:
# Test on a low-risk (strong) student profile
strong_student = {
    "age": 22, "distance_from_campus_km": 2.0, "midterm_score": 88.0,
    "assignment_score": 92.0, "lms_logins_per_week": 18, "video_watch_hours": 22.0,
    "forum_posts": 8, "quiz_attempts": 12, "avg_quiz_score": 90.0,
    "total_classes": 50, "classes_attended": 48, "attendance_rate": 96.0,
    "tardy_count": 0, "engagement_index": 88.0
}

result = predict_student(strong_student)
print("=== Prediction Result for Strong Student ===")
for k, v in result.items():
    print(f"{k}: {v}")

## Step 7: Model Performance Visualization

In [ ]:
# Feature importance for GPA predictor
importances = perf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Top Feature Importances - GPA Predictor", fontweight="bold")
plt.bar(range(10), importances[indices][:10], color="#6366f1")
plt.xticks(range(10), [feature_cols[i] for i in indices[:10]], rotation=45, ha="right")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

# Actual vs Predicted GPA
plt.figure(figsize=(8, 6))
plt.scatter(y_test, gpa_preds, alpha=0.3, c="#6366f1")
plt.plot([0, 4], [0, 4], "r--")
plt.xlabel("Actual GPA")
plt.ylabel("Predicted GPA")
plt.title(f"Actual vs Predicted GPA (R² = {r2:.3f})")
plt.tight_layout()
plt.show()

## Summary

The EduPredict notebook demonstrates the complete Big Data & ML pipeline:

| Stage | Technology | Output |
|:---|:---|:---|
| Data Ingestion | NumPy/Pandas | 4 raw datasets (5,000 students) |
| Preprocessing | Pandas | `edupredict_master_clean.csv` with derived features |
| Distributed Processing | PySpark | HDFS partitioned output, course demand summary |
| Model Training | scikit-learn | GPA, Dropout Risk, Anomaly models (.joblib) |
| Inference | Predict functions | Real-time risk assessment with confidence |

**Key business insights:**
- Attendance and LMS engagement are the strongest predictors of academic performance
- ~5% of students show anomalous academic patterns warranting intervention
- The pipeline scales from Colab (local mode) to full Hadoop cluster deployment
